In [8]:
import math
import sys
import yaml
import numpy as np
sys.path.append('../../python/')  
from periphery import logicGate
from periphery import constant
from periphery.Technology import Technology
from periphery.sramWriteDriver import SRAMWriteDriver
from periphery.precharger import Precharger
from periphery.WLdecoder import RowDecoder
from periphery.SenseAmp import SenseAmp
from periphery.DFF import DFF
from periphery.MUX import Mux
from periphery.levelShifter import LevelShifter
from periphery.WLDecoderDriver import WLNewDecoderDriver
from periphery.adder import Adder
from periphery.ADC import SarADC
from periphery.RNGBlock import RNG_block
from periphery.Bus import Bus
from simulator.subarray import SubArray
print(constant.INV)

0


In [9]:
with open('../../config.yaml', 'r') as file:
    config = yaml.safe_load(file)

with open('../../mapping.yaml', 'r') as file:
    mapping = yaml.safe_load(file)

with open('../../param.yaml', 'r') as file:
    param = yaml.safe_load(file)

with open('../../RNG.yaml', 'r') as file:
    RNG = yaml.safe_load(file)

In [10]:
class MAT:
    def __init__(self, tech,config, param, mapping, RNG, numSubArrayRow, numSubArrayCol, numCol, numRow, num_mu, num_sigma):

        self.tech = tech
        self.param = param
        self.config = config
        self.mapping = mapping
        self.RNG = RNG
        self.initialized = False
        self.num_col = numCol
        self.num_row = numRow
        self.numSubArrayRow = numSubArrayRow
        self.numSubArrayCol = numSubArrayCol
        self.num_mu = num_mu
        self.num_sigma = num_sigma

        self.feature_size = self.tech.get_param('featureSize')
        self.pnSizeRatio = self.tech.get_param('pnSizeRatio')
        self.vdd = self.tech.get_param('vdd')
        self.cell_type = self.config['device_type']
        self.clk_freq = self.config['frequency']
        self.temp = self.config['temperature']

        self.unitWireRes = self.param['unitLengthWireResistance']
        self.wireWidth = self.param['wireWidth']
        self.unitWireCap = 0.2e-15 / 1e-6  # 0.2 fF/um = 0.2e-15 F/micron

        self.resistanceOn = self.param['resistanceOn']
        self.resistanceOff = self.param['resistanceOff']
        self.resistanceAvg = (self.resistanceOn + self.resistanceOff) / 2
        self.writeVoltage = self.param['writeVoltage']
        self.readVoltage = self.param['readVoltage']
        self.accessVoltage = self.param['accessVoltage']
        self.avgWeightBit = self.param['cellBit']
        self.accesstype = self.param['accesstype']
        self.mem_mode = self.param['operationmode']
        self.readPulseWidth = self.param['readPulseWidth']
        self.heightInFeatureSizeSRAM = self.param['heightInFeatureSizeSRAM']
        self.widthInFeatureSizeSRAM = self.param['widthInFeatureSizeSRAM']
        self.widthSRAMCellNMOS = self.param['widthSRAMCellNMOS']
        self.widthSRAMCellPMOS = self.param['widthSRAMCellPMOS']
        self.widthAccessCMOS = self.param['widthAccessCMOS']
        self.minSenseVoltage = self.param['minSenseVoltage']
        self.heightInFeatureSize1T1R = self.param['heightInFeatureSize1T1R']
        self.heightInFeatureSizeCrossbar = self.param['heightInFeatureSizeCrossbar']
        self.widthInFeatureSize1T1R = self.param['widthInFeatureSize1T1R']
        self.widthInFeatureSizeCrossbar = self.param['widthInFeatureSizeCrossbar']

        if self.cell_type == 'SRAM':
            self.heightInFeatureSize = self.heightInFeatureSizeSRAM
            self.widthInFeatureSize = self.widthInFeatureSizeSRAM
        else:
            self.heightInFeatureSize = self.heightInFeatureSize1T1R if self.accessType == 'CMOS_access' else self.heightInFeatureSizeCrossbar

        self.SubArray = SubArray(numCol=self.num_col,numRow = self.num_row,num_mu= self.num_mu,num_sigma=self.num_sigma,array_x_overlap=0.2,
    array_y_overlap=0.4, relaxArrayCellWidth = False,relaxArrayCellHeight = False,tech=self.tech,config=config,mapping=self.mapping,param=param,RNG=self.RNG)
        self.SubArray_area, self.SubArray_height, self.SubArray_width,_ = self.SubArray.calculate_area()
        self.BusOutput = Bus(mode='VERTICAL',num_row=self.numSubArrayRow,num_col=self.numSubArrayCol,delay_tolerance=0,bus_width=64,unit_height=self.SubArray_height,unit_width=self.SubArray_width,clk_freq=self.clk_freq,param=param,config=config,tech=self.tech)

        self.initialized = True

    def calculate_area(self, overlap=False):
        if not self.initialized:
            raise ValueError("Bus must be initialized before calculating area.")

        BusOutput_area = self.BusOutput.calculate_area(folded_ratio=1.0, overlap=False)
        area = self.numSubArrayRow * self.numSubArrayCol * self.SubArray_area + BusOutput_area

        height = math.sqrt(area)
        width = area / height

        return area, height, width

    def calculate_latency(self, num_read):
        if not self.initialized:
            raise ValueError("Bus must be initialized before calculating latency.")
        SubArray_read_latency = self.SubArray.calculate_latency(calculate_clk_freq = self.clk_freq,validated=False)
        BusOutput_read_latency = self.BusOutput.calculate_latency(num_read = 1)
        read_latency = SubArray_read_latency[0] + BusOutput_read_latency
        return read_latency * num_read

    def calculate_power(self,input_vector,weight_matrix, num_bit_access, num_read):
        if not self.initialized:
            raise ValueError("Bus must be initialized before calculating power.")
        SubArray_read_energy,SubArray_write_energy,SubArray_leakage = self.SubArray.calculate_power(input_vector, weight_matrix)
        BusOutput_read_energy,BusOutput_leakage = self.BusOutput.calculate_power(num_bit_access = 64, num_read = 1)
        read_dynamic_energy = SubArray_read_energy + BusOutput_read_energy
        leakage = SubArray_leakage + BusOutput_leakage
        

        return read_dynamic_energy * num_bit_access * num_read, leakage



In [11]:
tech45 = Technology(node_nm=22, roadmap='LP')
# Instantiate and initialize Precharger
pre = MAT(
    numSubArrayRow=2,
    numSubArrayCol=2,
    numCol=128,
    numRow=128,
    num_mu=64,
    num_sigma=64,
    param=param,
    config=config,
    mapping=mapping,
    RNG=RNG,
    tech=tech45
)

lengthcol: 1.6984e-05
gatecap_senseamp_P: 2.3591700000000004e-16
junctioncap_senseamp_P: 1.0488957040349141e-16
gatecap_senseamp_N: 1.1795850000000002e-16
junctioncap_senseamp_N: 6.481047059118075e-17
gatecap_senseamp_P: 2.3591700000000004e-16
junctioncap_senseamp_P: 1.0488957040349141e-16
gatecap_senseamp_N: 1.1795850000000002e-16
junctioncap_senseamp_N: 6.481047059118075e-17


In [12]:
pre_charge_area, pre_height, pre_width = pre.calculate_area(overlap=False)

print("Area Result:", pre_charge_area)
print("Height Result:", pre_height)
print("Width Result:", pre_width)

Area Result: 4.7599651480509126e-08
Height Result: 0.0002181734435730186
Width Result: 0.0002181734435730186


In [13]:
read_latency = pre.calculate_latency(num_read=1)
print("Read Latency:", read_latency)

Read Latency: 4.59285743664082e-09


In [14]:
#######################################################################
weight_matrix = np.load('../../slice.npy')
# weight_matrix = np.load('../../conductance.npy')
# weight_matrix = 1 / weight_matrix
num_rows = weight_matrix.shape[0]
input_vector = [0] * num_rows
input_vector[4] = 1   
print("Input Vector Length:", len(input_vector))
#######################################################################

read_energy,leakage = pre.calculate_power(input_vector,weight_matrix, num_bit_access=32, num_read=1)
print(f"  Read Dynamic Energy: {read_energy:.3e} J")
print(f"  Leakage Power: {leakage:.3e} W")

Input Vector Length: 64
  Read Dynamic Energy: 3.099e-11 J
  Leakage Power: 5.319e-07 W
